In [1]:
using MarineHydro
using PyCall
cpt = pyimport("capytaine")

PyObject <module 'capytaine' from 'C:\\Users\\15183\\.julia\\conda\\3\\x86_64\\Lib\\site-packages\\capytaine\\__init__.py'>

## Create Multi DOF body using Capytaine

In [2]:
# Description of problem
h = Inf # sea depth [m]
omega = 1.03 # frequency [rad/s]
beta = 0 # incident wave angle [rad]
DOFs = ["Heave"] # degrees of freedom

1-element Vector{String}:
 "Heave"

In [3]:
# Create Mesh object
radius = 1.0 # 
resolution = (10, 10)
cptmesh = cpt.mesh_sphere(name="sphere", radius=radius, center=(0, 0, 0), resolution=resolution) 
cptmesh.keep_immersed_part(inplace=true)
println("Capytaine's Mesh object: ", cptmesh)

Capytaine's Mesh object: PyObject Mesh(vertices=[[... 51 vertices ...]], faces=[[... 50 faces ...]], name="sphere")


In [4]:
DOFs[1]

"Heave"

In [5]:
# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
foreach(dof -> cptbody.add_translation_dof(name=dof), DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Sphere 1"
cptbody.center_of_mass = (0, 0, 0)
cptbody.rotation_center = (0, 0, 0)
println("Capytaine's FloatingBody object: ", cptbody)

Capytaine's FloatingBody object: PyObject FloatingBody(mesh=Mesh(vertices=[[... 51 vertices ...]], faces=[[... 50 faces ...]], name="sphere"), lid_mesh=None, dofs={"Heave": ...}, center_of_mass=(0, 0, 0), name="Sphere 1")


In [6]:
cptbody.mesh.vertices

51×3 Matrix{Float64}:
 -1.0        1.22465e-16  -6.12323e-17
 -0.951057   1.16471e-16  -0.309017
 -0.809017  -0.587785     -6.12323e-17
 -0.809017   9.9076e-17   -0.587785
 -0.809017   0.587785     -6.12323e-17
 -0.769421  -0.559017     -0.309017
 -0.769421   0.559017     -0.309017
 -0.654508  -0.475528     -0.587785
 -0.654508   0.475528     -0.587785
 -0.587785   7.19829e-17  -0.809017
  ⋮                       
  0.654508  -0.475528     -0.587785
  0.654508   0.475528     -0.587785
  0.769421  -0.559017     -0.309017
  0.769421   0.559017     -0.309017
  0.809017  -0.587785     -6.12323e-17
  0.809017  -2.22045e-16  -0.587785
  0.809017   0.587785     -6.12323e-17
  0.951057  -2.22045e-16  -0.309017
  1.0       -2.22045e-16  -6.12323e-17

In [7]:
# Create BEMSolver object
solver = cpt.BEMSolver()

PyObject BEMSolver(engine=BasicMatrixEngine(linear_solver='lu_decomposition'), green_function=Delhommeau())

In [8]:
dof_list = cptbody.active_dofs

1-element Vector{String}:
 "Heave"

In [9]:
 # Create DiffractionProblem object
 diff_prob = cpt.DiffractionProblem(body=cptbody, wave_direction=beta, omega=omega, water_depth=h) 

PyObject DiffractionProblem(body=FloatingBody(..., name="Sphere 1"), omega=1.030, water_depth=inf, wave_direction=0.000)

In [10]:
# Create list of RadiationProblem objects
rad_prob = [] 

for dof in cptbody.active_dofs

    prob = cpt.RadiationProblem(
        body=cptbody, 
        omega=omega, 
        radiating_dof=dof, 
        water_depth=h
    )

    push!(rad_prob, prob) # this is a Julia way of appending p to the list rad_prob
end

In [11]:
# Solve diffraction and radiation problem
diff_result = solver.solve(diff_prob,keep_details=(true))
rad_result = solver.solve_all(rad_prob,keep_details=(true))
dataset = cpt.assemble_dataset([rad_result; diff_result])

Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.Dataset> Size: 194B
Dimensions:              (omega: 1, radiating_dof: 1, influenced_dof: 1,
                          wave_direction: 1)
Coordinates: (12/13)
  * omega                (omega) float64 8B 1.03
  * radiating_dof        (radiating_dof) category 9B Heave
  * influenced_dof       (influenced_dof) category 9B Heave
  * wave_direction       (wave_direction) float64 8B 0.0
    g                    float64 8B 9.81
    rho                  float64 8B 1e+03
    ...                   ...
    water_depth          float64 8B inf
    forward_speed        float64 8B 0.0
    freq                 (omega) float64 8B 0.1639
    period               (omega) float64 8B 6.1
    wavenumber           (omega) float64 8B 0.1081
    wavelength           (omega) float64 8B 58.1
Data variables:
    added_mass           (omega, radiating_dof, influenced_dof) float64 8B 1....
    radiation_damping    (omega, radiating_dof, influenced_dof) float64 8B 386.0
    diffraction_force    (omega, wave_direction, influenced_dof) complex128 16B ...
    Froude_Krylov_force  (omega, wave_direction, influenced_dof) complex128 16B ...
    excitation_force     (omega, wave_direction, influenced_dof) complex128 16B ...
Attributes:
    creation_of_dataset:  2026-02-06T16:47:07.478830
    capytaine_version:    2.3.1

In [12]:
# Get hydro_coeffs
num_dof = length(dataset.influenced_dof)
A_cpt = reshape(dataset.added_mass.values,num_dof,num_dof)
B_cpt = reshape(dataset.radiation_damping.values,num_dof,num_dof)
F_ex_cpt = reshape(dataset.excitation_force.values,num_dof,1)

println("Capytaine's Added Mass: ", A_cpt)
println("Capytaine's Radiation Damping: ", B_cpt)
println("Capytaine's Excitation Force: ", F_ex_cpt)

Capytaine's Added Mass: [1726.0582200595584;;]
Capytaine's Radiation Damping: [385.9705616048717;;]
Capytaine's Excitation Force: ComplexF64[25039.180423563783 - 399.1845125985226im;;]


## Create Multi DOF body using MarineHydro

In [13]:
# Create Mesh
mesh = Mesh(cptmesh)


Mesh([-1.0 1.2246467991473532e-16 -6.123233995736766e-17; -0.9510565162951535 1.1647083184890923e-16 -0.30901699437494745; … ; 0.9510565162951536 -2.220446049250313e-16 -0.30901699437494745; 1.0 -2.220446049250313e-16 -6.123233995736766e-17], [25 32 37 25; 40 41 37 32; … ; 46 36 34 44; 50 46 44 49], [0.18633899812498245 0.06054521066711338 -0.9673710108634358; 0.41864587972292056 0.13602629206872913 -0.8726779962499651; … ; 0.5454512662744504 -0.7507492613982619 -0.15321651614178625; 0.8825586880387303 -0.2867607008252568 -0.15321651614178625], [0.15623277758608398 0.05076310663489361 -0.9864149361260258; 0.4491304062125103 0.1459313151555306 -0.8814680535744689; … ; 0.5812281572696587 -0.7999919273347349 -0.1489521580782675; 0.9404469136807775 -0.3055697255163329 -0.14895215807826734], [0.028450753844463845, 0.0833531700197368, 0.13178202765372182, 0.16848421022922416, 0.028450753844463834, 0.08335317001973679, 0.1317820276537218, 0.16848421022922405, 0.028450753844463827, 0.083353170

In [14]:
# Helper function for rigid body dofs



In [15]:
dict = Dict("key1"=>1)
dict["key2"] = 2
dict

Dict{String, Int64} with 2 entries:
  "key2" => 2
  "key1" => 1

In [16]:
# Create FloatingBody
num_panels = mesh.nfaces

dof_mat_1 = zeros(num_panels,3)
dof_mat_1[:,3].=1

dof_mat_2 = zeros(num_panels,3)
dof_mat_2[:,2].=1

dofs = Dict("Heave"=>dof_mat_1, "Sway"=>dof_mat_2)


# struct FloatingBody
#     mesh::Mesh
#     dofs::Dict # {name::Str : dof::AbstractMatrix (nfaces,3)}

#     function FloatingBody(mesh::Mesh, dofs::Dict)
#         #add assert statements

#         return new(mesh, dofs)
#     end
# end



fb = FloatingBody(mesh,dofs)

FloatingBody(Mesh([-1.0 1.2246467991473532e-16 -6.123233995736766e-17; -0.9510565162951535 1.1647083184890923e-16 -0.30901699437494745; … ; 0.9510565162951536 -2.220446049250313e-16 -0.30901699437494745; 1.0 -2.220446049250313e-16 -6.123233995736766e-17], [25 32 37 25; 40 41 37 32; … ; 46 36 34 44; 50 46 44 49], [0.18633899812498245 0.06054521066711338 -0.9673710108634358; 0.41864587972292056 0.13602629206872913 -0.8726779962499651; … ; 0.5454512662744504 -0.7507492613982619 -0.15321651614178625; 0.8825586880387303 -0.2867607008252568 -0.15321651614178625], [0.15623277758608398 0.05076310663489361 -0.9864149361260258; 0.4491304062125103 0.1459313151555306 -0.8814680535744689; … ; 0.5812281572696587 -0.7999919273347349 -0.1489521580782675; 0.9404469136807775 -0.3055697255163329 -0.14895215807826734], [0.028450753844463845, 0.0833531700197368, 0.13178202765372182, 0.16848421022922416, 0.028450753844463834, 0.08335317001973679, 0.1317820276537218, 0.16848421022922405, 0.028450753844463827

In [17]:
# dof_mat_3 = zeros(num_panels,3)
# dof_mat_3[:,1].=1
# fb.dofs["Surge"] = dof_mat_3
# fb.dofs

In [18]:
dofs["Sway"]

50×3 Matrix{Float64}:
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 ⋮         
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0

In [19]:
# function radiation_bc(floatingbody::FloatingBody, omega)
#     """
        

#     Calculates the radiation boundary conditions for floating bodies at each panel.

#     # Arguments
#     - `floatingbody::FloatingBody`: The floating body
#     - `omega`: The frequency of the incident ocean wave ~~~.

#     # Returns
#     - The (Neumann) radiation boundary condition values for each panel.
# """
#     bc = Dict()
#     for dof_name in keys(floatingbody.dofs)
#         dof_mat = floatingbody.dofs[dof_name]
#         normals_mat = floatingbody.mesh.normals
#         bc[dof_name] = -1im .* omega .* sum(normals_mat .* dof_mat, dims=2)
#     end
#     return bc
# end


In [20]:
dof_name = "Sway"
omega = 1.3
radiation_bc(fb, omega)

Dict{Any, Any} with 2 entries:
  "Sway"  => ComplexF64[0.0-0.065992im; 0.0-0.189711im; … ; -0.0+1.03999im; -0.…
  "Heave" => ComplexF64[-0.0+1.28234im; -0.0+1.14591im; … ; -0.0+0.193638im; -0…

In [21]:
pressure = ones(mesh.nfaces,3)

50×3 Matrix{Float64}:
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 ⋮         
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0
 1.0  1.0  1.0

In [22]:
# function integrate_pressure(floatingbody::FloatingBody, pressure)
#   mesh = floatingbody.mesh
#   forces = Dict()
#   for dof_name in keys(floatingbody.dofs)
#     dof_mat = floatingbody.dofs[dof_name]
#     normal_dof_amp_on_face = -sum(dof_mat .* mesh.normals, dims=2)
#     forces[dof_name] = sum(pressure .* normal_dof_amp_on_face .* mesh.areas)
#   end
#   return forces
# end

In [23]:
integrate_pressure(fb, pressure)

Dict{Any, Any} with 2 entries:
  "Sway"  => 7.63278e-17
  "Heave" => 8.81678

In [24]:
A,B = calculate_radiation_forces(fb, omega)


(Dict((1.3, "Sway", "Sway") => 1148.4127878444547, (1.3, "Heave", "Heave") => 1584.41887703679, (1.3, "Sway", "Heave") => -3.3635159089235507e-13, (1.3, "Heave", "Sway") => 3.573735653231273e-14), Dict((1.3, "Sway", "Sway") => 13.33423468890405, (1.3, "Heave", "Heave") => 638.3292353104384, (1.3, "Sway", "Heave") => -1.5303997385602156e-13, (1.3, "Heave", "Sway") => 1.0675221390626504e-16))

In [25]:
A

Dict{Tuple{Float64, String, String}, Float64} with 4 entries:
  (1.3, "Sway", "Sway")   => 1148.41
  (1.3, "Heave", "Heave") => 1584.42
  (1.3, "Sway", "Heave")  => -3.36352e-13
  (1.3, "Heave", "Sway")  => 3.57374e-14

In [26]:
F_D = DiffractionForce(fb,omega)

Dict{Tuple{Float64, String}, ComplexF64} with 2 entries:
  (1.3, "Sway")  => -2.4869e-13+3.37508e-13im
  (1.3, "Heave") => -2554.36-815.949im

In [27]:
F_FK = FroudeKrylovForce(fb, omega)

Dict{Tuple{Float64, String}, ComplexF64} with 2 entries:
  (1.3, "Sway")  => 9.09495e-13-1.42109e-14im
  (1.3, "Heave") => 25709.2-1.63425e-13im

In [28]:
# Old code
# ω = 1.03
# ζ = [0,0,1] # HEAVE
# F = DiffractionForce(mesh,ω,ζ)
# A,B = calculate_radiation_forces(mesh,ζ,ω)

In [29]:
rigid_dof_list = ["Heave", "Pitch"]
rotation_center = [0,0,1]
fb2 = FloatingBody(mesh, rigid_dof_list, rotation_center)

FloatingBody(Mesh([-1.0 1.2246467991473532e-16 -6.123233995736766e-17; -0.9510565162951535 1.1647083184890923e-16 -0.30901699437494745; … ; 0.9510565162951536 -2.220446049250313e-16 -0.30901699437494745; 1.0 -2.220446049250313e-16 -6.123233995736766e-17], [25 32 37 25; 40 41 37 32; … ; 46 36 34 44; 50 46 44 49], [0.18633899812498245 0.06054521066711338 -0.9673710108634358; 0.41864587972292056 0.13602629206872913 -0.8726779962499651; … ; 0.5454512662744504 -0.7507492613982619 -0.15321651614178625; 0.8825586880387303 -0.2867607008252568 -0.15321651614178625], [0.15623277758608398 0.05076310663489361 -0.9864149361260258; 0.4491304062125103 0.1459313151555306 -0.8814680535744689; … ; 0.5812281572696587 -0.7999919273347349 -0.1489521580782675; 0.9404469136807775 -0.3055697255163329 -0.14895215807826734], [0.028450753844463845, 0.0833531700197368, 0.13178202765372182, 0.16848421022922416, 0.028450753844463834, 0.08335317001973679, 0.1317820276537218, 0.16848421022922405, 0.028450753844463827

In [30]:
fb2.dofs["Pitch"]

50×3 adjoint(::Matrix{Float64}) with eltype Float64:
 -1.96737  0.0  -0.186339
 -1.87268  0.0  -0.418646
 -1.69256  0.0  -0.636992
 -1.44465  0.0  -0.797729
 -1.96737  0.0  -0.115164
 -1.87268  0.0  -0.258737
 -1.69256  0.0  -0.393683
 -1.44465  0.0  -0.493023
 -1.96737  0.0  -4.62593e-18
 -1.87268  0.0   0.0
  ⋮             
 -1.15322  0.0  -0.545451
 -1.15322  0.0   9.20712e-18
 -1.15322  0.0   0.545451
 -1.15322  0.0   0.882559
 -1.15322  0.0   0.882559
 -1.15322  0.0   0.545451
 -1.15322  0.0   2.94628e-16
 -1.15322  0.0  -0.545451
 -1.15322  0.0  -0.882559

In [31]:
A,B = calculate_radiation_forces(fb2, omega)

(Dict((1.3, "Heave", "Heave") => 1584.41887703679, (1.3, "Heave", "Pitch") => 3.3635159089235507e-14, (1.3, "Pitch", "Pitch") => 1217.5650198287938, (1.3, "Pitch", "Heave") => 7.56791079507799e-14), Dict((1.3, "Heave", "Heave") => 638.3292353104384, (1.3, "Heave", "Pitch") => 1.120898246015783e-14, (1.3, "Pitch", "Pitch") => 14.125209702778236, (1.3, "Pitch", "Heave") => 2.022313940240285e-13))

In [32]:
F_D = DiffractionForce(fb2,omega)

Dict{Tuple{Float64, String}, ComplexF64} with 2 entries:
  (1.3, "Pitch") => -15.3819+1714.07im
  (1.3, "Heave") => -2554.36-815.949im

In [33]:
F_FK = FroudeKrylovForce(fb2, omega)

Dict{Tuple{Float64, String}, ComplexF64} with 2 entries:
  (1.3, "Pitch") => 1.81899e-12+3111.06im
  (1.3, "Heave") => 25709.2-1.63425e-13im